# MLP-TUNE-002 — архитектура и функции активации

Это отдельное исследование после 140 trials `MLP-TUNE-001`. Оно не изменяет
первую SQLite study и не создаёт официальные `MLP-xxx` эксперименты.

Optuna подбирает архитектуру, функцию активации и параметры обучения. Первая
trial воспроизводит лучший screening-вариант №86 с `ReLU`. После screening
несколько кандидатов обязательно проверяются на пяти официальных folds.

`test.csv` здесь не используется. После работы завершите kernel через
**Kernel → Shut Down Kernel**, чтобы освободить память.


## 1. Ограничиваем потоки

Выполните эту ячейку первой. Trials идут последовательно, а библиотеки линейной
алгебры не создают лишние рабочие потоки.


In [1]:
import os

for variable in (
    "OPENBLAS_NUM_THREADS", "OMP_NUM_THREADS", "OMP_THREAD_LIMIT",
    "MKL_NUM_THREADS", "NUMEXPR_NUM_THREADS", "BLIS_NUM_THREADS",
):
    os.environ[variable] = "1"


## 2. Данные и фиксированные screening folds

Используются те же признаки и fold-safe преобразования, что у принятой MLP-024.
Все trials видят одинаковые три folds с seed 2026, поэтому их можно сравнивать
между собой.


In [2]:
from dataclasses import replace
from pathlib import Path
import sys

import optuna
import pandas as pd
import torch
from IPython.display import display

torch.set_num_threads(1)

CURRENT_DIR = Path.cwd().resolve()
PROJECT_ROOT = next(
    path for path in (CURRENT_DIR, *CURRENT_DIR.parents)
    if (path / "README.md").is_file() and (path / "src").is_dir()
)
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from ml_project import DataCatalog
import ml_project.config as project_config
import ml_project.mlp_experiment as mlp_tools
import ml_project.mlp_tuning as tuning

PARENT_MODULE = "ml_project.mlp_experiments.mlp_024_dpoutmedium"
parent = mlp_tools.load_mlp_experiment(PARENT_MODULE)

catalog = DataCatalog(PROJECT_ROOT, project_config.RAW_DIR, project_config.DATASETS)
raw_train = catalog.load(project_config.TRAIN_DATASET)
prepared = parent.prepare_features(raw_train.copy(deep=True))
mlp_tools.validate_feature_data(
    prepared, raw_train, target=project_config.TARGET, key=project_config.KEY
)

build_fold_transformer = getattr(parent, "build_fold_transformer", None)
screening_splits = tuning.make_stratified_splits(
    prepared,
    target=project_config.TARGET,
    n_splits=3,
    random_state=2026,
)

print("Project:", PROJECT_ROOT)
print("Parent:", parent.EXPERIMENT.experiment_id, parent.EXPERIMENT.title)
print("Rows:", len(prepared.frame))
print("Screening folds:", len(screening_splits))


C:\ML_Progs\Runtimes\Miniforge3\envs\titanik-ml\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project: D:\Projects\Titanik-ML-project
Parent: MLP-024 dpoutmedium
Rows: 891
Screening folds: 3


## 3. Что выбирает Optuna

Каждая сеть использует одну функцию активации во всех скрытых слоях:
`ReLU`, `LeakyReLU`, `GELU`, `SiLU` или `SquaredReLU`. Для `LeakyReLU`
дополнительно выбирается `negative_slope`.

Также меняются:

- `batch_size`: 32 или 64;
- `learning_rate`: от 0.0012 до 0.0025;
- `weight_decay`: от 1e-7 до 1e-4;
- `dropout`: от 0.05 до 0.30;
- число скрытых слоёв: от 3 до 5;
- общая ширина скрытых слоёв: 4, 8, 12 или 16.

Dropout остаётся после последнего скрытого слоя. BatchNorm не добавляется.


In [3]:
STUDY_NAME = "titanic_mlp_tuning_002"
STUDY_DIR = PROJECT_ROOT / "artifacts/optuna"
STUDY_DIR.mkdir(parents=True, exist_ok=True)
DATABASE_PATH = STUDY_DIR / f"{STUDY_NAME}.db"
STORAGE = f"sqlite:///{DATABASE_PATH.as_posix()}"

study = optuna.create_study(
    study_name=STUDY_NAME,
    storage=STORAGE,
    load_if_exists=True,
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=43),
)

if not study.trials:
    study.enqueue_trial(tuning.PHASE2_BASELINE_PARAMETERS)

objective = tuning.make_phase2_objective(
    prepared,
    target=project_config.TARGET,
    key=project_config.KEY,
    base_spec=parent.EXPERIMENT,
    cv_splits=screening_splits,
    build_fold_transformer=build_fold_transformer,
)

print("Storage:", DATABASE_PATH.relative_to(PROJECT_ROOT))
print("Existing trials:", len(study.trials))
display(pd.Series(tuning.PHASE2_BASELINE_PARAMETERS, name="trial 86 + ReLU").to_frame())


[I 2026-10-05 14:35:18,679] A new study created in RDB with name: titanic_mlp_tuning_002


Storage: artifacts\optuna\titanic_mlp_tuning_002.db
Existing trials: 1


,trial 86 + ReLU
batch_size,32
learning_rate,0.00175
weight_decay,0.0001
dropout,0.3
n_layers,4
hidden_dim,8
activation,relu


## 4. Запускаем небольшими этапами

Начните с 20 trials. Повторный запуск этой ячейки добавит ещё 20 в ту же study.
После 60–80 trials проверьте, продолжает ли улучшаться максимум и какие активации
встречаются среди лучших результатов.


In [7]:
N_TRIALS = 500

study.optimize(
    objective,
    n_trials=N_TRIALS,
    n_jobs=1,
    gc_after_trial=True,
    show_progress_bar=True,
)


Best trial: 0. Best value: 0.832772:   0%|          | 1/500 [00:05<43:50,  5.27s/it]

[I 2026-10-05 14:53:11,870] Trial 130 finished with value: 0.8249158249158249 and parameters: {'activation': 'leaky_relu', 'batch_size': 32, 'learning_rate': 0.0015053842021964775, 'weight_decay': 0.0001, 'dropout': 0.05, 'n_layers': 3, 'hidden_dim': 4, 'negative_slope': 0.05}. Best is trial 0 with value: 0.8327721661054994.


Best trial: 0. Best value: 0.832772:   0%|          | 2/500 [00:11<49:36,  5.98s/it]

[I 2026-10-05 14:53:18,335] Trial 131 finished with value: 0.8282828282828283 and parameters: {'activation': 'leaky_relu', 'batch_size': 32, 'learning_rate': 0.0013526328397794084, 'weight_decay': 1e-06, 'dropout': 0.1, 'n_layers': 3, 'hidden_dim': 4, 'negative_slope': 0.05}. Best is trial 0 with value: 0.8327721661054994.


Best trial: 0. Best value: 0.832772:   1%|          | 3/500 [00:17<50:02,  6.04s/it]

...

[I 2026-10-05 15:42:13,944] Trial 627 finished with value: 0.8271604938271606 and parameters: {'activation': 'leaky_relu', 'batch_size': 32, 'learning_rate': 0.0013554407844808898, 'weight_decay': 1e-05, 'dropout': 0.25, 'n_layers': 3, 'hidden_dim': 4, 'negative_slope': 0.05}. Best is trial 0 with value: 0.8327721661054994.


Best trial: 0. Best value: 0.832772: 100%|█████████▉| 499/500 [49:14<00:06,  6.09s/it]

[I 2026-10-05 15:42:21,397] Trial 628 finished with value: 0.8237934904601572 and parameters: {'activation': 'relu', 'batch_size': 64, 'learning_rate': 0.0012007313769975244, 'weight_decay': 1e-06, 'dropout': 0.15000000000000002, 'n_layers': 3, 'hidden_dim': 4}. Best is trial 0 with value: 0.8327721661054994.


Best trial: 0. Best value: 0.832772: 100%|██████████| 500/500 [49:20<00:00,  5.92s/it]

[I 2026-10-05 15:42:26,859] Trial 629 finished with value: 0.8237934904601572 and parameters: {'activation': 'relu', 'batch_size': 64, 'learning_rate': 0.0020405718123590036, 'weight_decay': 1e-06, 'dropout': 0.1, 'n_layers': 3, 'hidden_dim': 4}. Best is trial 0 with value: 0.8327721661054994.


## 5. Таблица результатов

`value` — средняя accuracy на трёх screening folds. Сохраняем таблицу в CSV,
чтобы результаты не зависели от состояния notebook.


In [8]:
trials = study.trials_dataframe()
csv_path = STUDY_DIR / f"{STUDY_NAME}_trials.csv"
trials.to_csv(csv_path, index=False)

columns = [
    "number", "value", "state",
    "params_activation", "params_negative_slope",
    "params_batch_size", "params_learning_rate", "params_weight_decay",
    "params_dropout", "params_n_layers", "params_hidden_dim",
    "user_attrs_accuracy_std", "user_attrs_log_loss_mean",
]
available = [column for column in columns if column in trials]
complete = trials.loc[trials["state"].eq("COMPLETE"), available]
display(complete.sort_values("value", ascending=False).head(15))

print("Saved:", csv_path.relative_to(PROJECT_ROOT))
print("Best screening accuracy:", study.best_value)
print("Best parameters:", study.best_params)


,number,value,state,params_activation,params_negative_slope,params_batch_size,params_learning_rate,params_weight_decay,params_dropout,params_n_layers,params_hidden_dim,user_attrs_accuracy_std,user_attrs_log_loss_mean
0,0,0.832772,COMPLETE,relu,NaN,32,0.001750,1.000000e-04,0.3,4,8,0.023888,0.454552
443,443,0.830527,COMPLETE,leaky_relu,0.05,32,0.002466,1.000000e-04,0.1,3,4,0.031285,0.423690
137,137,0.830527,COMPLETE,leaky_relu,0.05,32,0.001285,1.000000e-07,0.1,3,4,0.028036,0.422700
193,193,0.830527,COMPLETE,leaky_relu,0.05,32,0.001367,1.000000e-04,0.1,3,4,0.028036,0.422808
201,201,0.830527,COMPLETE,leaky_relu,0.05,32,0.001367,1.000000e-04,0.1,3,4,0.028036,0.422806
256,256,0.830527,COMPLETE,leaky_relu,0.05,32,0.001336,1.000000e-05,0.1,3,4,0.028036,0.422081
284,284,0.830527,COMPLETE,leaky_relu,0.05,32,0.001313,1.000000e-05,0.1,3,4,0.028036,0.422366
414,414,0.830527,COMPLETE,leaky_relu,0.05,32,0.001294,1.000000e-07,0.1,3,4,0.028036,0.422340
272,272,0.830527,COMPLETE,leaky_relu,0.05,32,0.001325,1.000000e-05,0.1,3,4,0.028036,0.422021
209,209,0.829405,COMPLETE,relu,NaN,32,0.001361,1.000000e-04,0.1,3,4,0.026153,0.423262


Saved: artifacts\optuna\titanic_mlp_tuning_002_trials.csv
Best screening accuracy: 0.8327721661054994
Best parameters: {'activation': 'relu', 'batch_size': 32, 'learning_rate': 0.0017503534730658975, 'weight_decay': 0.0001, 'dropout': 0.3, 'n_layers': 4, 'hidden_dim': 8}


## 6. Смотрим не только на одну лучшую trial

Одна trial может случайно удачно попасть в три folds. Здесь сравниваются число
попаданий каждой активации в top-20, её лучшая accuracy и медиана результатов.


In [9]:
complete_all = trials.loc[trials["state"].eq("COMPLETE")].copy()
top_20 = complete_all.nlargest(min(20, len(complete_all)), "value")

activation_summary = (
    complete_all.groupby("params_activation")
    .agg(
        trials=("number", "count"),
        best_accuracy=("value", "max"),
        median_accuracy=("value", "median"),
        mean_accuracy=("value", "mean"),
    )
    .sort_values("best_accuracy", ascending=False)
)
top_counts = top_20["params_activation"].value_counts().rename("top_20_count")
display(activation_summary.join(top_counts).fillna({"top_20_count": 0}))


,trials,best_accuracy,median_accuracy,mean_accuracy,top_20_count
params_activation,,,,,
relu,262,0.832772,0.822671,0.821335,5.0
leaky_relu,288,0.830527,0.823793,0.822617,15.0
gelu,31,0.824916,0.817059,0.815430,0.0
silu,26,0.822671,0.814815,0.815592,0.0
squared_relu,23,0.819304,0.750842,0.758210,0.0


## 7. Официальная перепроверка трёх лучших trials

Эта ячейка ничего не сохраняет в реестр и не принимает решение `adopt`. Она
заново обучает MLP-024 и три лучших кандидата на пяти folds с seed 42. Запускайте
её после окончания screening.

Если несколько лучших trials почти одинаковы, замените `head(3)` на выбранные
номера, чтобы проверить разные активации.


In [10]:
official_splits = tuning.make_stratified_splits(
    prepared,
    target=project_config.TARGET,
    n_splits=5,
    random_state=42,
)

reference_spec = replace(
    parent.EXPERIMENT,
    experiment_id="TUNE2-OFFICIAL-REFERENCE",
    title="MLP-024 official-fold reference",
    save_artifacts=False,
    sync_docs=False,
)
reference_result = mlp_tools.run_mlp_cv(
    prepared,
    target=project_config.TARGET,
    key=project_config.KEY,
    build_network=parent.build_network,
    spec=reference_spec,
    build_fold_transformer=build_fold_transformer,
    cv_splits=official_splits,
)

rows = [{
    "candidate": "MLP-024",
    "screening_trial": None,
    "accuracy": float(reference_result.summary.loc["accuracy", "mean"]),
    "accuracy_std": float(reference_result.summary.loc["accuracy", "std"]),
    "balanced_accuracy": float(reference_result.summary.loc["balanced_accuracy", "mean"]),
    "f1": float(reference_result.summary.loc["f1", "mean"]),
    "roc_auc": float(reference_result.summary.loc["roc_auc", "mean"]),
    "log_loss": float(reference_result.summary.loc["log_loss", "mean"]),
}]

selected_trials = sorted(
    (trial for trial in study.trials if trial.value is not None),
    key=lambda trial: trial.value,
    reverse=True,
)[:3]

for selected in selected_trials:
    parameters = selected.params
    candidate_spec = replace(
        parent.EXPERIMENT,
        experiment_id=f"TUNE2-OFFICIAL-{selected.number:04d}",
        title=f"Official-fold check of trial {selected.number}",
        training=tuning.training_from_parameters(parent.EXPERIMENT.training, parameters),
        save_artifacts=False,
        sync_docs=False,
    )
    result = mlp_tools.run_mlp_cv(
        prepared,
        target=project_config.TARGET,
        key=project_config.KEY,
        build_network=tuning.build_phase2_network_factory(parameters),
        spec=candidate_spec,
        build_fold_transformer=build_fold_transformer,
        cv_splits=official_splits,
    )
    rows.append({
        "candidate": f"trial {selected.number}: {parameters['activation']}",
        "screening_trial": selected.number,
        "accuracy": float(result.summary.loc["accuracy", "mean"]),
        "accuracy_std": float(result.summary.loc["accuracy", "std"]),
        "balanced_accuracy": float(result.summary.loc["balanced_accuracy", "mean"]),
        "f1": float(result.summary.loc["f1", "mean"]),
        "roc_auc": float(result.summary.loc["roc_auc", "mean"]),
        "log_loss": float(result.summary.loc["log_loss", "mean"]),
    })

official_comparison = pd.DataFrame(rows).sort_values("accuracy", ascending=False)
display(official_comparison)


,candidate,screening_trial,accuracy,accuracy_std,balanced_accuracy,f1,roc_auc,log_loss
0,MLP-024,NaN,0.838365,0.011105,0.817573,0.775582,0.878035,0.405802
2,trial 137: leaky_relu,137.0,0.823778,0.013175,0.801956,0.754702,0.874080,0.411377
3,trial 193: leaky_relu,193.0,0.823778,0.014323,0.800263,0.752603,0.874057,0.412888
1,trial 0: relu,0.0,0.819296,0.009417,0.799265,0.751380,0.871340,0.429972


## 8. Как принять решение

Победа на screening folds только выдвигает кандидата. Если он сохраняет улучшение
на пяти официальных folds, создайте обычный `new-MLPexperiment` на основе MLP-024,
перенесите выбранную архитектуру и выполните стандартный notebook. Только его
paired-сравнение и карточка используются для решения `adopt` или `reject`.

Не выбирайте seed как гиперпараметр: это поиск удачной случайности, а не модели.
